In [296]:
pip install arch yfinance

In [297]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import yfinance as yf
from arch import arch_model
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

Fetching Data

In [298]:
nifty50 = [
    'RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'HINDUNILVR.NS',
    'ICICIBANK.NS', 'SBIN.NS', 'BHARTIARTL.NS', 'ITC.NS', 'ASIANPAINT.NS',
    'MARUTI.NS', 'HCLTECH.NS', 'KOTAKBANK.NS', 'LT.NS', 'AXISBANK.NS',
    'TITAN.NS', 'SUNPHARMA.NS', 'ULTRACEMCO.NS', 'NESTLEIND.NS', 'WIPRO.NS',
    'JSWSTEEL.NS', 'M&M.NS', 'NTPC.NS', 'POWERGRID.NS', 'TATAMOTORS.NS',
    'TECHM.NS', 'HDFCLIFE.NS', 'SBILIFE.NS', 'BAJFINANCE.NS', 'COALINDIA.NS',
    'HINDALCO.NS', 'GRASIM.NS', 'BRITANNIA.NS', 'SHREECEM.NS', 'CIPLA.NS',
    'DIVISLAB.NS', 'DRREDDY.NS', 'EICHERMOT.NS', 'HEROMOTOCO.NS', 'BAJAJFINSV.NS',
    'INDUSINDBK.NS', 'TATASTEEL.NS', 'ADANIENT.NS', 'APOLLOHOSP.NS', 'BPCL.NS',
    'ONGC.NS', 'TATACONSUM.NS', 'BAJAJ-AUTO.NS', 'ADANIPORTS.NS', 'UPL.NS'
]
# nifty50 = [
#           'RELIANCE.NS', 'TCS.NS','HDFCBANK.NS']
data= yf.download(nifty50,start='2022-06-01',end='2025-06-01',interval="1d")
data=data["Close"]
data = data.rename(columns={ticker: f'{ticker}' for ticker in nifty50})
data = data.dropna()
data

[*********************100%***********************]  50 of 50 completed


Ticker,ADANIENT.NS,ADANIPORTS.NS,APOLLOHOSP.NS,ASIANPAINT.NS,AXISBANK.NS,BAJAJ-AUTO.NS,BAJAJFINSV.NS,BAJFINANCE.NS,BHARTIARTL.NS,BPCL.NS,...,SUNPHARMA.NS,TATACONSUM.NS,TATAMOTORS.NS,TATASTEEL.NS,TCS.NS,TECHM.NS,TITAN.NS,ULTRACEMCO.NS,UPL.NS,WIPRO.NS
Date,,,,,,,,,,,,,,,,,,,,,
2022-06-01,2146.948242,720.333008,3801.658203,2751.374756,683.197388,3365.660400,1257.137207,590.485840,678.923035,141.601379,...,812.466492,736.402405,439.440399,93.607559,3194.407959,1056.330078,2169.134033,5842.608398,749.363403,225.300476
2022-06-02,2205.340820,731.623291,3609.792969,2805.437744,688.674500,3337.251709,1291.009155,595.172180,680.193665,143.056458,...,832.118652,733.644897,434.053619,95.191811,3258.958740,1055.777344,2192.181396,5874.091797,745.134888,226.927689
2022-06-03,2185.976318,723.265564,3553.891846,2784.554932,674.483765,3322.776123,1266.618164,592.259216,670.957581,142.535233,...,837.491455,729.436096,426.887756,93.937614,3275.286865,1057.527710,2175.845703,5552.416504,739.657043,227.573792
2022-06-06,2220.962158,724.585205,3598.354980,2717.567383,669.404968,3453.327881,1249.133301,591.551819,667.487976,138.712875,...,832.360535,738.869690,427.332550,94.888168,3266.528076,1054.119019,2170.071777,5457.087402,748.642639,226.473038
2022-06-07,2223.457520,718.035828,3574.192627,2647.396240,663.081299,3468.798828,1231.398804,577.561279,669.784668,139.103775,...,821.179260,732.048401,430.791931,94.043228,3201.548340,1041.313354,2073.587891,5431.520020,717.120850,223.816803
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-05-26,2546.397949,1393.420654,7096.500000,2305.925049,1214.162354,8737.068359,2051.600586,922.518677,1832.083984,318.159180,...,1670.600586,1137.866821,722.785583,158.806534,3494.277832,1572.665039,3599.467529,11615.374023,626.318420,245.412842
2025-05-27,2539.001709,1397.401367,7073.500000,2306.321533,1194.579102,8741.457031,2028.911499,912.715759,1828.513916,312.297424,...,1677.876709,1130.024902,710.342590,157.975922,3456.147217,1563.826538,3578.630127,11349.061523,624.633789,243.461227
2025-05-28,2513.714600,1404.168457,6956.500000,2282.144531,1193.679810,8630.274414,2021.415161,922.319641,1840.711792,316.927734,...,1660.932495,1113.150024,711.829834,157.565491,3455.159424,1551.943604,3563.974365,11167.213867,625.030212,242.657028


Log_returns and expected return

In [309]:
log_returns = np.log(data/data.shift(1)).dropna()
expected_return= log_returns.mean()

EGARCH Modelling

In [300]:
rolling_window= 90
forecasted_vols = []

for ticker in nifty50:
        new_returns= log_returns[ticker].dropna()
        dates= new_returns.loc["2022-06-01":"2024-06-01"].index

        for current_date in dates:
            current_idx = new_returns.index.get_loc(current_date)

            if current_idx <= rolling_window:
                continue

            train_data= new_returns.iloc[current_idx-rolling_window:current_idx]

            try:
                model= arch_model(train_data, vol='EGARCH', p=1, q=1, dist='normal')
                result= model.fit(disp='off')
                forecast= result.forecast(horizon=1)
                variance= forecast.variance.iloc[-1, 0]
                volatility= np.sqrt(variance)
            except Exception as e:
                volatility = np.nan

            forecasted_vols.append({
                'ticker': ticker,
                'date': current_date,
                'forecast_vol': volatility
            })

forecast_df = pd.DataFrame(forecasted_vols)
forecast_df.dropna(inplace=True)
forecast_df.set_index(['ticker', 'date'], inplace=True)
forecast_df

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.11/dist-packages/arch/univariate/base.py:309: DataScaleWarning:

y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.0003332. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.


/usr/local/lib/python3.11/dist-packages/arch/univariate/base.py:309: DataScaleWarning:

y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.0003303. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.


/usr/local/lib/python3.11/dist-packages/arch/univariate/base.

forecast_vol
ticker      date                    
RELIANCE.NS 2022-10-13  1.510112e-39
            2022-10-14  2.229903e-33
            2022-10-17  1.873983e+00
            2022-10-18  4.609124e-78
            2022-10-19  3.794277e-76
...                              ...
UPL.NS      2024-05-27  2.194877e-02
            2024-05-28  2.295371e-02
            2024-05-29  2.239059e-02
            2024-05-30  2.117223e-02
            2024-05-31  2.329009e-02

[20050 rows x 1 columns]

Scoring and selecting

In [301]:
risk_aversion=0.5
avg_volatility= forecast_df.groupby('ticker')['forecast_vol'].mean()
scores = (1 - risk_aversion) * expected_return - risk_aversion * avg_volatility
selected_stocks = scores.sort_values(ascending=False).head(2).index.tolist()
print("Top 2 selected stocks:", selected_stocks)

Top 2 selected stocks: ['SUNPHARMA.NS', 'INFY.NS']


Fetching data for selected stocks

In [302]:
tickers= selected_stocks
data1= yf.download(tickers,start='2024-06-01',end='2025-06-01',interval="1d")
data1=data1["Close"]
data1 = data1.rename(columns={ticker: f'{ticker}' for ticker in tickers})
data1 = data1.dropna()

[*********************100%***********************]  2 of 2 completed


Strategy(MACD) for selected stocks

In [303]:
pd.options.mode.chained_assignment = None    ### hiding warnings
def strategy(data,stock,signal):
    ema_fast= data[stock].ewm(span=12).mean()
    ema_slow= data[stock].ewm(span=26).mean()
    macd=ema_fast-ema_slow
    signal_line= macd.ewm(span=9).mean()
    data[signal]=0
    i=0
    while i<len(data):
      if macd[i]>signal_line[i]:
        data[signal].iloc[i]=1
      elif macd[i]<signal_line[i]:
        data[signal].iloc[i]=-1
      i+=1

    i=0
    while i < len(data) and data[signal].iloc[i]!=1:  ### removing all sells before first buy.
      data[signal].iloc[i]=0
      i+=1

    i=0
    while i<len(data)-1:            ### Keeping one trade active at a time....removing repetitive buy and sell...
      if data[signal].iloc[i]==-1:
        j=i+1
        while j<len(data) and data[signal].iloc[j]!=1:
          data[signal].iloc[j]=0
          j+=1
      i+=1
    i=0
    while i<len(data)-1:
      if data[signal].iloc[i]==1:
        j=i+1
        while j<len(data) and data[signal].iloc[j]!=-1:
          data[signal].iloc[j]=0
          j+=1
      i+=1
    return data

i=0
while i<2:
  strategy(data1,selected_stocks[i],f'signal{i+1}')
  i+=1
data1

Ticker,INFY.NS,SUNPHARMA.NS,signal1,signal2
Date,,,,
2024-06-03,1370.785400,1435.055786,0,0
2024-06-04,1358.841309,1412.143066,0,0
2024-06-05,1394.380859,1468.881592,1,1
2024-06-06,1435.478271,1454.462402,0,0
2024-06-07,1495.295776,1488.189453,0,0
...,...,...,...,...
2025-05-26,1558.570679,1670.600586,0,0
2025-05-27,1548.315063,1677.876709,0,0
2025-05-28,1549.991455,1660.932495,0,0


In [304]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=data1.index, y=data1[selected_stocks[0]], name = selected_stocks[0], line = dict(color='blue', width=1.5)))
fig.add_trace(go.Scatter(x=data1.index, y=data1[selected_stocks[1]], name = selected_stocks[1], line = dict(color='black', width=1.5)))
fig.add_trace(go.Scatter(x=data1.index[data1['signal1']==1], y=data1[selected_stocks[0]][data1['signal1']==1], name = "Buy Signal", mode = 'markers', marker = dict(symbol = 'triangle-up', color='green', size=8)))
fig.add_trace(go.Scatter(x=data1.index[data1['signal1']==-1], y=data1[selected_stocks[0]][data1['signal1']==-1], name = "Sell Signal", mode = 'markers', marker = dict(symbol = 'triangle-down', color='red', size=8)))
fig.add_trace(go.Scatter(x=data1.index[data1['signal1']==1], y=data1[selected_stocks[1]][data1['signal1']==1], name = "Buy Signal", mode = 'markers', marker = dict(symbol = 'triangle-up', color='green', size=8)))
fig.add_trace(go.Scatter(x=data1.index[data1['signal1']==-1], y=data1[selected_stocks[1]][data1['signal1']==-1], name = "Sell Signal", mode = 'markers', marker = dict(symbol = 'triangle-down', color='red', size=8)))
fig.update_layout(title=dict(text="SIGNALS",x=0.45,y=0.9,font=dict(size=30)),yaxis_title="Price",xaxis_title="Date")
fig.show()

Backtesting by giving equal weights to both

In [305]:
initial_capital=100000
i=0
while i<2:
    data1[f'return{i+1}'] = data1[selected_stocks[1]].pct_change()
    Strategy_return1 = data1[f'return{i+1}'] * data1[f'signal{i+1}']
    data1[f'capital{i+1}'] = (1 + Strategy_return1).cumprod()*(initial_capital / 2)
    i+=1

finding cumulative return for selected stocks

In [306]:
R_user= ((data1["capital1"][-1]+data1["capital2"][-1])-initial_capital)/initial_capital*100
R_user

np.float64(20.958220900060958)

finding cumulative return for nifty50 ( just holding the market )

In [307]:
start = "2024-06-01"
end = "2025-06-01"
initial_capital=100000
final_capital = []
i=0
while i<len(nifty50):
    stock_data = data[nifty50[i]].loc[start:end]
    data[f'return{i+1}'] = stock_data.pct_change()
    data_capital = (1 +data[f'return{i+1}']).cumprod()*(initial_capital / len(nifty50))
    final_capital.append(data_capital[-1])
    i+=1
R_market=(np.sum(final_capital)-initial_capital)/initial_capital*100
R_market

np.float64(5.9639716874750155)

User Score %

In [308]:
user_score= (R_user/R_market)*100
print(user_score,'%')

351.41382284016345 %
